# 00 — Cohere-assisted Study Builder contract

This notebook tests one narrow product hypothesis:

> Can Cohere help a researcher turn structural evidence and explicit answers into a draft study contract without becoming the study executor?

The language model may propose questions and a schema-valid draft. It may not approve scientific meaning, execute the study, or change the contract after approval. FeatureGraph will eventually execute only a researcher-approved, fingerprinted contract.

This first notebook uses a small, network-free fixture shaped like the completed PhysioNet wearable study. It contains no source participant data and does not reproduce the maintained 33-participant result.


## 0. Install and configure

From the repository root:

```bash
python -m pip install -e ".[dev,notebooks,study-builder]"
export COHERE_API_KEY="your-key"
python -m jupyterlab notebooks/study_builder/00_cohere_assisted_contract.ipynb
```

The notebook runs in offline demonstration mode by default. Review the local evidence packet before changing `RUN_COHERE` to `True`. The API key is read from the environment and is never printed or written into an artifact.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import JSON, display
from jsonschema import Draft202012Validator

COHERE_MODEL = "command-a-plus-05-2026"
RUN_COHERE = False
RESEARCHER_APPROVES = False
PROMPT_TEMPLATE_VERSION = "study-builder-questions-v0.1"
COHERE_API_KEY = os.environ.get("COHERE_API_KEY")


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the FeatureGraph repository.")


REPO_ROOT = find_repo_root()
SCHEMA_PATH = (
    REPO_ROOT
    / "notebooks"
    / "study_builder"
    / "study_contract_v0.schema.json"
)
print(f"Repository: {REPO_ROOT}")
print(f"Cohere requests enabled: {RUN_COHERE}")
print(f"API key available: {bool(COHERE_API_KEY)}")


## 1. Create a local, PhysioNet-shaped fixture

These tables exercise the structural questions the Study Builder must ask: participant identity, protocol version, external tags, one-to-one self-reports, and sensor streams with separate native time axes.

They deliberately omit enough scientific meaning that the authoring assistant must ask questions instead of silently completing the study.


In [ ]:
fixture_files = {
    "participants.csv": pd.DataFrame(
        {
            "participant_id": ["demo_v1", "demo_v2"],
            "protocol_version": ["version_1", "version_2"],
        }
    ),
    "tags.csv": pd.DataFrame(
        {
            "participant_id": ["demo_v1"] * 5 + ["demo_v2"] * 5,
            "tag_order": list(range(5)) * 2,
            "timestamp_seconds": [0.0, 60.0, 120.0, 180.0, 240.0] * 2,
        }
    ),
    "self_reports.csv": pd.DataFrame(
        {
            "participant_id": ["demo_v1", "demo_v1", "demo_v2", "demo_v2"],
            "protocol_state": ["baseline", "task", "baseline", "task"],
            "self_reported_stress": [1, 6, 2, 7],
        }
    ),
    "heart_rate.csv": pd.DataFrame(
        {
            "participant_id": ["demo_v1"] * 4 + ["demo_v2"] * 4,
            "timestamp_seconds": [0.0, 30.0, 60.0, 90.0] * 2,
            "heart_rate": [64.0, 66.0, 72.0, 70.0, 62.0, 65.0, 74.0, 71.0],
        }
    ),
    "eda.csv": pd.DataFrame(
        {
            "participant_id": ["demo_v1"] * 4 + ["demo_v2"] * 4,
            "timestamp_seconds": [0.0, 15.0, 30.0, 45.0] * 2,
            "eda": [0.20, 0.22, 0.31, 0.29, 0.18, 0.21, 0.34, 0.30],
        }
    ),
    "temperature.csv": pd.DataFrame(
        {
            "participant_id": ["demo_v1"] * 4 + ["demo_v2"] * 4,
            "timestamp_seconds": [0.0, 20.0, 40.0, 60.0] * 2,
            "temperature": [32.1, 32.2, 32.3, 32.2, 31.9, 32.0, 32.2, 32.1],
        }
    ),
}

PROTOCOL_NOTES = [
    "The published study contains two protocol versions with different ordered stages.",
    "Physical button marks in tags.csv are the external candidate boundaries.",
    "Version 1 documentation and executable indexing disagree about one final tag.",
    "Time not assigned by a declared protocol stage must not be given a new label.",
    "Self-reported stress is an external measurement, not a label inferred from sensors.",
    "Heart rate, EDA, and temperature must remain at their native sampling rates.",
    "Known exclusions must retain the source-provided reason.",
]

pd.DataFrame(
    {
        "file_id": fixture_files.keys(),
        "rows": [len(frame) for frame in fixture_files.values()],
        "columns": [", ".join(frame.columns) for frame in fixture_files.values()],
    }
)


## 2. Profile locally before contacting a model

The profiler sends structure, not full sensor streams. Identifier values are redacted. In a future product, public documentation and explicitly selected samples could be added separately.


In [ ]:
SENSITIVE_NAME_PARTS = ("participant", "subject", "patient", "person", "id")


def json_scalar(value: Any) -> Any:
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value


def profile_frame(file_id: str, frame: pd.DataFrame) -> dict[str, Any]:
    columns = []
    for name in frame.columns:
        series = frame[name]
        sensitive = any(part in name.lower() for part in SENSITIVE_NAME_PARTS)
        column = {
            "name": name,
            "dtype": str(series.dtype),
            "missing_count": int(series.isna().sum()),
            "unique_count": int(series.nunique(dropna=True)),
            "distinct_preview": (
                ["<redacted>"]
                if sensitive
                else [json_scalar(value) for value in series.dropna().unique()[:5]]
            ),
        }
        if pd.api.types.is_numeric_dtype(series) and not sensitive:
            column["minimum"] = json_scalar(series.min())
            column["maximum"] = json_scalar(series.max())
        columns.append(column)
    return {
        "file_id": file_id,
        "row_count": int(len(frame)),
        "columns": columns,
    }


evidence_packet = {
    "fixture_notice": (
        "Synthetic structural fixture only; do not infer cohort counts or "
        "scientific results from these rows."
    ),
    "protocol_notes": PROTOCOL_NOTES,
    "file_profiles": [
        profile_frame(file_id, frame)
        for file_id, frame in fixture_files.items()
    ],
}

evidence_json = json.dumps(evidence_packet, sort_keys=True, indent=2)
evidence_sha256 = hashlib.sha256(evidence_json.encode()).hexdigest()
print(f"Evidence fingerprint: {evidence_sha256}")
display(JSON(evidence_packet))


Before enabling Cohere, inspect the packet above. It should contain file structure and declared documentation facts, but no raw identifier values and no invented meaning for the protocol marks.


## 3. Load the draft contract schema

Structured output guarantees shape, not scientific truth. Local validation will therefore check both the JSON Schema and whether every referenced file and column exists in the evidence packet.


In [ ]:
study_contract_schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
Draft202012Validator.check_schema(study_contract_schema)

clarification_schema = {
    "type": "object",
    "additionalProperties": False,
    "required": ["observed_facts", "blocked_assumptions", "questions"],
    "properties": {
        "observed_facts": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["evidence", "implication"],
                "properties": {
                    "evidence": {"type": "string"},
                    "implication": {"type": "string"},
                },
            },
        },
        "blocked_assumptions": {
            "type": "array",
            "items": {"type": "string"},
        },
        "questions": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["id", "question", "why_required"],
                "properties": {
                    "id": {"type": "string"},
                    "question": {"type": "string"},
                    "why_required": {"type": "string"},
                },
            },
        },
    },
}

Draft202012Validator.check_schema(clarification_schema)
print("Both JSON Schemas are valid.")


## 4. Define the bounded Cohere call

The model receives no tools and cannot execute code. It returns one JSON object constrained by the supplied schema.


In [ ]:
SYSTEM_MESSAGE = """You assist a researcher in authoring a FeatureGraph study contract.
Separate observed evidence, proposed interpretation, and unresolved questions.
Never assign scientific meaning to unlabeled boundaries, gaps, sensor changes,
or missing records. Never claim clinical validity, causality, or detection.
Return only the requested JSON object."""

QUESTION_PROMPT = """Generate a JSON clarification packet matching the supplied schema.
Ask only questions whose answers materially change grouping, boundaries, joins,
exclusions, measurements, validation, or claim limits. Do not draft a final
contract yet.

Evidence packet:
{evidence}
"""


def call_cohere_json(
    *,
    prompt: str,
    schema: dict[str, Any],
    prompt_version: str,
) -> tuple[dict[str, Any], dict[str, Any]]:
    if not COHERE_API_KEY:
        raise RuntimeError(
            "Set COHERE_API_KEY in the environment before enabling Cohere."
        )

    import cohere

    api_schema = {
        key: value
        for key, value in schema.items()
        if key not in {"$schema", "title"}
    }
    client = cohere.ClientV2(api_key=COHERE_API_KEY)
    response = client.chat(
        model=COHERE_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": prompt},
        ],
        response_format={
            "type": "json_object",
            "json_schema": api_schema,
        },
        temperature=0,
    )
    response_text = response.message.content[0].text
    payload = json.loads(response_text)
    Draft202012Validator(schema).validate(payload)

    provenance = {
        "mode": "cohere",
        "model": COHERE_MODEL,
        "cohere_sdk_version": cohere.__version__,
        "prompt_template_version": prompt_version,
        "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
        "schema_sha256": hashlib.sha256(
            json.dumps(api_schema, sort_keys=True).encode()
        ).hexdigest(),
        "response_sha256": hashlib.sha256(response_text.encode()).hexdigest(),
        "response_id": getattr(response, "id", None),
    }
    return payload, provenance


## 5. Ask for clarification questions

Offline mode uses a frozen example so that the notebook remains testable and reviewable without spending credits.


In [ ]:
OFFLINE_CLARIFICATION_PACKET = {
    "observed_facts": [
        {
            "evidence": "tags.csv contains ordered external timestamps.",
            "implication": "The timestamps may bound protocol occurrences.",
        },
        {
            "evidence": "The sensor files have separate timestamp columns.",
            "implication": "Each stream can retain its native time axis.",
        },
    ],
    "blocked_assumptions": [
        "Do not assign meaning to the extra version-1 tag.",
        "Do not label gaps as rest or recovery without a declaration.",
        "Do not infer stress from heart rate, EDA, or temperature.",
    ],
    "questions": [
        {
            "id": "extra_version_1_tag",
            "question": (
                "Should the final version-1 tag remain uninterpreted because "
                "the source executable indexing does not use it?"
            ),
            "why_required": "Its meaning is not declared consistently.",
        },
        {
            "id": "undeclared_time",
            "question": (
                "Should time outside named protocol stages remain unassigned?"
            ),
            "why_required": "Assigning a state would add undeclared meaning.",
        },
        {
            "id": "self_report_join",
            "question": (
                "Must each protocol occurrence join to exactly one source "
                "self-report by participant and protocol state?"
            ),
            "why_required": "The expected join cardinality must be explicit.",
        },
    ],
}

question_prompt = QUESTION_PROMPT.format(evidence=evidence_json)

if RUN_COHERE:
    clarification_packet, question_provenance = call_cohere_json(
        prompt=question_prompt,
        schema=clarification_schema,
        prompt_version=PROMPT_TEMPLATE_VERSION,
    )
else:
    clarification_packet = OFFLINE_CLARIFICATION_PACKET
    question_provenance = {
        "mode": "offline_frozen_example",
        "prompt_template_version": PROMPT_TEMPLATE_VERSION,
    }

display(JSON(clarification_packet))
display(pd.DataFrame(clarification_packet["questions"]))


## 6. Record researcher answers

Edit this cell. The values below reproduce the decisions already documented in the maintained PhysioNet study; they are not model-generated scientific conclusions.


In [ ]:
RESEARCHER_ANSWERS = {
    "extra_version_1_tag": (
        "Yes. Preserve it as a source boundary but leave it uninterpreted."
    ),
    "undeclared_time": "Yes. Leave time outside named stages unassigned.",
    "self_report_join": (
        "Yes. Require a one-to-one join by participant_id and protocol_state."
    ),
    "protocol_version_1_states": [
        "baseline",
        "stroop",
        "rest_1",
        "tmct",
        "rest_2",
        "real_opinion",
        "opposite_opinion",
        "subtraction",
    ],
    "protocol_version_2_states": [
        "baseline",
        "tmct",
        "rest_1",
        "real_opinion",
        "opposite_opinion",
        "rest_2",
        "subtraction",
    ],
    "exclusions": [
        {"participant": "S02", "reason": "duplicated raw signals"},
        {
            "participant": "f07",
            "reason": "PPG and temperature sensors covered by protection dock",
        },
        {
            "participant": "f14",
            "reason": "protocol split across recordings after Bluetooth loss",
        },
    ],
}

display(JSON(RESEARCHER_ANSWERS))


## 7. Draft the study contract

A real Cohere draft is allowed to contain unresolved questions. It is not eligible for approval until the schema and all deterministic checks pass and `unresolved_questions` is empty.


In [ ]:
CONTRACT_PROMPT_VERSION = "study-builder-contract-v0.1"

CONTRACT_PROMPT = """Generate a JSON FeatureGraph study contract matching the
supplied schema. Use only the evidence packet and researcher answers. Do not
invent files, columns, protocol stages, exclusions, joins, measurements, or
scientific interpretations. Put any unsupported decision in
unresolved_questions.

Evidence packet:
{evidence}

Researcher answers:
{answers}
"""

OFFLINE_DRAFT_CONTRACT = {
    "contract_version": "study-contract-v0",
    "study": {
        "name": "PhysioNet wearable protocol representation",
        "dataset": "Wearable Device Dataset from Induced Stress and Structured Exercise Sessions",
        "dataset_version": "1.0.1",
        "unit_of_analysis": "declared protocol occurrence",
        "claim_boundaries": [
            "Does not detect stress.",
            "Does not establish causality.",
            "Does not validate a physiological biomarker.",
        ],
    },
    "sources": [
        {
            "file_id": "participants.csv",
            "role": "participants",
            "required_columns": ["participant_id", "protocol_version"],
        },
        {
            "file_id": "tags.csv",
            "role": "boundaries",
            "required_columns": [
                "participant_id",
                "tag_order",
                "timestamp_seconds",
            ],
        },
        {
            "file_id": "self_reports.csv",
            "role": "self_reports",
            "required_columns": [
                "participant_id",
                "protocol_state",
                "self_reported_stress",
            ],
        },
        {
            "file_id": "heart_rate.csv",
            "role": "sensor_stream",
            "required_columns": [
                "participant_id",
                "timestamp_seconds",
                "heart_rate",
            ],
        },
        {
            "file_id": "eda.csv",
            "role": "sensor_stream",
            "required_columns": [
                "participant_id",
                "timestamp_seconds",
                "eda",
            ],
        },
        {
            "file_id": "temperature.csv",
            "role": "sensor_stream",
            "required_columns": [
                "participant_id",
                "timestamp_seconds",
                "temperature",
            ],
        },
    ],
    "participants": {
        "source_file": "participants.csv",
        "id_column": "participant_id",
        "protocol_version_column": "protocol_version",
    },
    "time_axes": [
        {
            "source_file": "tags.csv",
            "time_column": "timestamp_seconds",
            "units": "seconds",
            "sampling_policy": "external_boundaries",
        },
        {
            "source_file": "heart_rate.csv",
            "time_column": "timestamp_seconds",
            "units": "seconds",
            "sampling_policy": "preserve_native_rate",
        },
        {
            "source_file": "eda.csv",
            "time_column": "timestamp_seconds",
            "units": "seconds",
            "sampling_policy": "preserve_native_rate",
        },
        {
            "source_file": "temperature.csv",
            "time_column": "timestamp_seconds",
            "units": "seconds",
            "sampling_policy": "preserve_native_rate",
        },
    ],
    "protocol_versions": [
        {
            "version": "version_1",
            "participant_rule": "protocol_version equals version_1",
            "ordered_states": RESEARCHER_ANSWERS[
                "protocol_version_1_states"
            ],
        },
        {
            "version": "version_2",
            "participant_rule": "protocol_version equals version_2",
            "ordered_states": RESEARCHER_ANSWERS[
                "protocol_version_2_states"
            ],
        },
    ],
    "boundaries": {
        "source_file": "tags.csv",
        "interpretation": (
            "Use only button marks declared by the source executable protocol "
            "mapping; preserve unused marks without assigning meaning."
        ),
        "undeclared_policy": "leave_unassigned",
    },
    "joins": [
        {
            "source_file": "self_reports.csv",
            "target": "protocol_occurrences",
            "keys": ["participant_id", "protocol_state"],
            "cardinality": "one_to_one",
        }
    ],
    "exclusions": RESEARCHER_ANSWERS["exclusions"],
    "measurements": [
        {
            "source_file": "heart_rate.csv",
            "signal_column": "heart_rate",
            "statistics": ["count", "mean", "minimum", "maximum"],
        },
        {
            "source_file": "eda.csv",
            "signal_column": "eda",
            "statistics": ["count", "mean", "minimum", "maximum"],
        },
        {
            "source_file": "temperature.csv",
            "signal_column": "temperature",
            "statistics": ["count", "mean", "minimum", "maximum"],
        },
    ],
    "validations": [
        {
            "id": "source_boundaries_preserved",
            "assertion": "Every declared start and end equals its source tag.",
        },
        {
            "id": "self_reports_one_to_one",
            "assertion": "Every occurrence joins to exactly one self-report.",
        },
        {
            "id": "native_rates_preserved",
            "assertion": "Sensor streams are not interpolated or resampled.",
        },
        {
            "id": "undeclared_time_unassigned",
            "assertion": "No undeclared gap receives a protocol-state label.",
        },
    ],
    "unresolved_questions": [],
}

contract_prompt = CONTRACT_PROMPT.format(
    evidence=evidence_json,
    answers=json.dumps(RESEARCHER_ANSWERS, sort_keys=True, indent=2),
)

if RUN_COHERE:
    draft_contract, contract_provenance = call_cohere_json(
        prompt=contract_prompt,
        schema=study_contract_schema,
        prompt_version=CONTRACT_PROMPT_VERSION,
    )
else:
    draft_contract = OFFLINE_DRAFT_CONTRACT
    contract_provenance = {
        "mode": "offline_frozen_example",
        "prompt_template_version": CONTRACT_PROMPT_VERSION,
    }

display(JSON(draft_contract))


## 8. Validate structure and evidence references

Passing these checks means that the proposal is internally consistent with the evidence packet. It does not mean that its scientific interpretation is correct.


In [ ]:
def referenced_columns(contract: dict[str, Any]) -> list[tuple[str, str]]:
    references = []
    for source in contract["sources"]:
        references.extend(
            (source["file_id"], column)
            for column in source["required_columns"]
        )
    participants = contract["participants"]
    references.extend(
        [
            (participants["source_file"], participants["id_column"]),
            (
                participants["source_file"],
                participants["protocol_version_column"],
            ),
        ]
    )
    references.extend(
        (axis["source_file"], axis["time_column"])
        for axis in contract["time_axes"]
    )
    references.extend(
        (measurement["source_file"], measurement["signal_column"])
        for measurement in contract["measurements"]
    )
    return references


def validate_against_evidence(
    contract: dict[str, Any],
    profiles: list[dict[str, Any]],
) -> pd.DataFrame:
    checks: list[dict[str, Any]] = []

    schema_errors = sorted(
        Draft202012Validator(study_contract_schema).iter_errors(contract),
        key=lambda error: list(error.path),
    )
    checks.append(
        {
            "check": "json_schema",
            "passed": not schema_errors,
            "detail": (
                "valid"
                if not schema_errors
                else "; ".join(error.message for error in schema_errors[:3])
            ),
        }
    )

    columns_by_file = {
        profile["file_id"]: {
            column["name"] for column in profile["columns"]
        }
        for profile in profiles
    }
    declared_files = {source["file_id"] for source in contract["sources"]}
    observed_files = set(columns_by_file)

    checks.append(
        {
            "check": "declared_files_exist",
            "passed": declared_files <= observed_files,
            "detail": str(sorted(declared_files - observed_files)),
        }
    )

    missing_references = [
        f"{file_id}:{column}"
        for file_id, column in referenced_columns(contract)
        if file_id not in columns_by_file
        or column not in columns_by_file[file_id]
    ]
    checks.append(
        {
            "check": "referenced_columns_exist",
            "passed": not missing_references,
            "detail": str(missing_references),
        }
    )

    empty_reasons = [
        exclusion["participant"]
        for exclusion in contract["exclusions"]
        if not exclusion["reason"].strip()
    ]
    checks.append(
        {
            "check": "exclusions_have_reasons",
            "passed": not empty_reasons,
            "detail": str(empty_reasons),
        }
    )

    unresolved = contract["unresolved_questions"]
    checks.append(
        {
            "check": "no_unresolved_questions",
            "passed": not unresolved,
            "detail": str(unresolved),
        }
    )

    one_to_one = all(
        join["cardinality"] == "one_to_one"
        for join in contract["joins"]
        if join["source_file"] == "self_reports.csv"
    )
    checks.append(
        {
            "check": "self_report_join_is_one_to_one",
            "passed": one_to_one,
            "detail": "required by researcher answer",
        }
    )

    return pd.DataFrame(checks)


validation = validate_against_evidence(
    draft_contract,
    evidence_packet["file_profiles"],
)
validation


In [ ]:
assert validation["passed"].all(), validation.loc[~validation["passed"]]
print("All deterministic draft checks passed.")


## 9. Approve only after inspection

Leave `RESEARCHER_APPROVES = False` while experimenting. Setting it to `True` records the approved contract fingerprint. The fingerprint protects the decision boundary; it does not make Cohere's proposal scientifically authoritative.


In [ ]:
canonical_contract = json.dumps(
    draft_contract,
    sort_keys=True,
    separators=(",", ":"),
)
candidate_sha256 = hashlib.sha256(canonical_contract.encode()).hexdigest()

if RESEARCHER_APPROVES:
    if not validation["passed"].all():
        raise ValueError("A contract with failed checks cannot be approved.")
    approved_contract_sha256 = candidate_sha256
    print(f"Approved contract SHA-256: {approved_contract_sha256}")
else:
    approved_contract_sha256 = None
    print(f"Candidate contract SHA-256: {candidate_sha256}")
    print("Not approved. Review the draft and researcher answers first.")


## What you have built

This notebook now demonstrates the complete authoring boundary:

1. profile study files locally;
2. present structural evidence without raw identifiers;
3. ask Cohere for schema-constrained clarification questions;
4. record researcher answers separately;
5. ask Cohere for a draft contract;
6. validate every file and column reference deterministically;
7. require explicit approval before fingerprinting.

It deliberately stops before dataset execution.

### Your first run

1. Run the notebook once with `RUN_COHERE = False`.
2. Read the evidence packet and the three clarification questions.
3. Edit one value in `RESEARCHER_ANSWERS` and rerun the draft and validation cells.
4. Export `COHERE_API_KEY`, restart the kernel, and set `RUN_COHERE = True`.
5. Compare Cohere's questions and draft with the frozen offline example.
6. Do **not** set `RESEARCHER_APPROVES = True` until you can explain every field.

The repository's execution layer now loads a fingerprinted, approved PhysioNet contract and protects the 33-participant, 248-occurrence, and 99-check results. This notebook remains the assisted authoring layer; the deterministic runner performs execution without Cohere.
